# Kiki SLM — SFT Fine-tuning on Google Colab

**Prerequisites (done locally):**
1. `python scripts/prepare_sft_chatml.py` — download & format 10 SFT datasets
2. Upload `kiki_sft_chatml_train.jsonl` and `kiki_sft_chatml_eval.jsonl` to `My Drive/kiki-slm/data/`
3. Upload `gold_100.jsonl` to `My Drive/kiki-slm/data/` (for evaluation)

**What this notebook does:**
- Calls `scripts/colab_train.py` — trains Qwen3-4B with QLoRA, logs to W&B
- Calls `scripts/colab_eval.py` — compares base vs fine-tuned on gold data
- All processing lives in version-controlled scripts, not in this notebook

## 1. Setup

In [ ]:
%%capture
# Install uv, then all deps via uv
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

!uv pip install --system unsloth wandb datasets

In [ ]:
# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone / update repo and set working directory
import os

REPO_DIR = "/content/drive/MyDrive/kiki-slm/repo"

if os.path.exists(f"{REPO_DIR}/.git"):
    !cd {REPO_DIR} && git fetch origin && git reset --hard origin/main
    print(f"Updated repo at {REPO_DIR}")
else:
    !git clone https://github.com/BlueSpringsAI/kiki-slm.git {REPO_DIR}
    print(f"Cloned repo to {REPO_DIR}")

%cd {REPO_DIR}
!ls scripts/colab_*.py

In [ ]:
# ============================================================
# CONFIGURE PATHS — edit these if your Drive layout differs
# ============================================================
DRIVE_DATA = "/content/drive/MyDrive/kiki-slm/data"
DRIVE_OUT  = "/content/drive/MyDrive/kiki-slm/adapters/kiki-sft-v1"

TRAIN_FILE = f"{DRIVE_DATA}/kiki_sft_chatml_train.jsonl"
EVAL_FILE  = f"{DRIVE_DATA}/kiki_sft_chatml_eval.jsonl"
GOLD_FILE  = f"{DRIVE_DATA}/gold_100.jsonl"

# Verify files exist
for label, path in [("Train", TRAIN_FILE), ("Eval", EVAL_FILE)]:
    assert os.path.exists(path), f"{label} file not found: {path}"
    print(f"{label}: {os.path.getsize(path)/1024/1024:.1f} MB")

if os.path.exists(GOLD_FILE):
    print(f"Gold:  {sum(1 for l in open(GOLD_FILE) if l.strip())} tickets")
else:
    print(f"Gold:  not found (will use 5-example template)")

In [ ]:
# W&B login (interactive — needs notebook)
import wandb
wandb.login()

In [ ]:
# Quick data inspection
from datasets import load_dataset
from collections import Counter

ds = load_dataset("json", data_files=TRAIN_FILE, split="train")
print(f"Train: {len(ds):,} examples, columns={ds.column_names}")

if "source" in ds.column_names:
    for src, cnt in sorted(Counter(ds['source']).items(), key=lambda x: -x[1]):
        print(f"  {src:<30s} {cnt:>6,} ({cnt/len(ds)*100:.1f}%)")

del ds  # free memory

## 2. Train

Runs `scripts/colab_train.py` as a subprocess with `python -u` (unbuffered).
- Real-time **tqdm progress bar** + loss logs visible below
- Loss curves, LR schedule, GPU stats tracked in **W&B dashboard**
- Correct step count printed at training start (accounts for packing + grad accum)

In [ ]:
!python -u scripts/colab_train.py \
    --train-file {TRAIN_FILE} \
    --eval-file {EVAL_FILE} \
    --output-dir {DRIVE_OUT} \
    --epochs 3 \
    --lr 2e-4 \
    --lora-r 32 \
    --lora-alpha 64 \
    --wandb \
    --wandb-project kiki-slm

In [ ]:
# Display training summary + W&B link
import json
metrics_path = f"{DRIVE_OUT}/training_metrics.json"
if os.path.exists(metrics_path):
    m = json.load(open(metrics_path))
    print(f"Final loss:  {m.get('final_loss', 'N/A')}")
    print(f"Duration:    {m.get('duration_hours', 'N/A')}h")
    print(f"Peak VRAM:   {m.get('peak_vram_gb', 'N/A')}GB")
    if m.get('wandb_url'):
        print(f"\nW&B Dashboard: {m['wandb_url']}")
else:
    print("Training metrics not found — check output above for errors")

## 3. Evaluate — Base vs Fine-tuned

Loads base Qwen3 and fine-tuned model **sequentially** (one at a time) to avoid OOM.
Strips `<think>` tokens before JSON parsing.

In [ ]:
!python -u scripts/colab_eval.py \
    --adapter-path {DRIVE_OUT} \
    --gold-file {GOLD_FILE} \
    --output-file {DRIVE_OUT}/eval_results.json

In [ ]:
# Quick inference test with fine-tuned model
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(DRIVE_OUT, max_seq_length=2048, load_in_4bit=True)
FastLanguageModel.for_inference(model)

test_messages = [
    "My order #ORD-48293 hasn't arrived yet. I placed it 5 days ago.",
    "I received a damaged laptop. The screen is cracked. I want a full refund.",
    "Someone made unauthorized purchases on my account!",
]

system = "You are Kiki, an AI customer service agent. Respond with valid JSON: intent, urgency, workflow_steps, tools_required, reasoning, response."

for msg in test_messages:
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": system}, {"role": "user", "content": msg}],
        tokenize=False, add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"Customer: {msg}")
    print(f"-"*60)
    print(f"Kiki: {resp[:500]}")

## Done!

**Adapter saved at:** `My Drive/kiki-slm/adapters/kiki-sft-v1/`

**Next steps (locally):**
1. Download adapter from Google Drive
2. Place at `outputs/adapters/kiki-sft-v1/`
3. Full 4-way eval: `python scripts/3_evaluate.py --model-path outputs/adapters/kiki-sft-v1`
4. Demo: `python scripts/4_demo.py --model outputs/adapters/kiki-sft-v1`

**W&B:** Check your dashboard for full training curves, LR schedule, and GPU metrics.